In [7]:

import pandas as pd
import polars as pl
from pathlib import Path

data_dir = Path("../data/raw")

files = {
    "salaries": "american_soccer_analysis_mls_salaries_players.csv",
    "goals_added_gk": "american_soccer_analysis_mls_goals-added_goalkeepers.csv",
    "goals_added": "american_soccer_analysis_mls_goals-added_players.csv",
    "xgoals": "american_soccer_analysis_mls_xgoals_players.csv",
    "xpass": "american_soccer_analysis_mls_xpass_players.csv",
}

asa_tables = {
    source: pd.read_csv(data_dir / filename)
    for source, filename in files.items()
}

for source, df in asa_tables.items():
    print(f"\n--- {source} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    if "Season" in df.columns:
        print(f"Seasons: {sorted(df['Season'].dropna().unique().tolist())}")



--- salaries ---
Shape: (15490, 8)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Base Salary', 'Guaranteed Compensation', 'Date']
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

--- goals_added_gk ---
Shape: (572, 12)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Minutes', 'Claiming', 'Fielding', 'Handling', 'Passing', 'Shotstopping', 'Sweeping', 'Goals Added']
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

--- goals_added ---
Shape: (6399, 13)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Minutes', 'Dribbling', 'Fouling', 'Interrupting', 'Passing', 'Receiving', 'Shooting', 'Goals Added']
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

--- xgoals ---
Shape: (6970, 20)
Columns: ['Unnamed: 0', 'Player', 'Team', 'Season', 'Position', 'Minutes', 'Shots', 'SoT', 'G', 'xG', 'xPlace', 'G-xG', 'KeyP', 'A', 'xA', 'A-xA', 'G+A', 'xG+xA', 'PA', 'xPA']
Seasons: [2016, 2017, 201

In [8]:

keys = ["Player", "Season"]
results = []

for source, df in asa_tables.items():
    historical = df[df["Season"].between(2016, 2025)]

    results.append({
        "source": source,
        "rows": len(historical),
        "unique_player_seasons": (
            historical[keys].drop_duplicates().shape[0]
        ),
        "duplicate_player_season_rows": (
            historical.duplicated(subset=keys).sum()
        ),
        "missing_player": historical["Player"].isna().sum(),
        "missing_season": historical["Season"].isna().sum(),
        "unique_players": historical["Player"].nunique(),
    })

coverage_structure = pd.DataFrame(results)

display(coverage_structure)


,source,rows,unique_player_seasons,duplicate_player_season_rows,missing_player,missing_season,unique_players
0,salaries,14579,8196,6383,0,0,2677
1,goals_added_gk,572,572,0,0,0,162
2,goals_added,6399,6397,2,0,0,2239
3,xgoals,6970,6968,2,0,0,2400
4,xpass,6970,6968,2,0,0,2400


In [9]:

keys = ["Player", "Season"]

# Unique player-seasons in salaries
salary_population = (
    asa_tables["salaries"]
    .loc[
        lambda df: df["Season"].between(2016, 2025),
        keys
    ]
    .drop_duplicates()
)

# Combine all four performance sources
performance_population = pd.concat(
    [
        df.loc[
            df["Season"].between(2016, 2025),
            keys
        ]
        for source, df in asa_tables.items()
        if source != "salaries"
    ],
    ignore_index=True
).drop_duplicates()

# Find salary player-seasons without a performance match
salary_coverage = salary_population.merge(
    performance_population.assign(has_performance=True),
    on=keys,
    how="left",
    validate="one_to_one"
)

salary_only = (
    salary_coverage[
        salary_coverage["has_performance"].isna()
    ]
    .drop(columns="has_performance")
    .sort_values(["Season", "Player"])
    .reset_index(drop=True)
)

print(f"Salary player-seasons: {len(salary_population):,}")
print(f"Performance player-seasons: {len(performance_population):,}")
print(f"Salary-only player-seasons: {len(salary_only):,}")
print(f"Unique salary-only players: {salary_only['Player'].nunique():,}")

display(salary_only.head(20))


Salary player-seasons: 8,196
Performance player-seasons: 6,968
Salary-only player-seasons: 1,349
Unique salary-only players: 870


,Player,Season
0,Aaron Long,2016
1,Alejandro Zendejas,2016
2,Alex Tambakis,2016
3,Andre Rawls,2016
4,Andrea Mancini,2016
5,Andrés Romero,2016
6,Andy Craven,2016
7,Andy Thoma,2016
8,Anthony Manning,2016
9,Auston Trusty,2016


In [10]:

players_to_check = ["Andre Rawls", "Auston Trusty"]

for player in players_to_check:
    print(f"\n--- {player} ---")

    for source, df in asa_tables.items():
        records = df.loc[
            df["Player"].eq(player)
            & df["Season"].between(2016, 2025)
        ]

        print(f"\n{source}: {len(records)} rows")

        if not records.empty:
            columns = [
                col for col in
                ["Player", "Team", "Season", "Position", "Minutes", "Date"]
                if col in records.columns
            ]

            display(
                records[columns]
                .sort_values("Season")
                .drop_duplicates()
            )



--- Andre Rawls ---

salaries: 11 rows


,Player,Team,Season,Position,Date
14876,Andre Rawls,NYC,2016,GK,2016-05-15
14877,Andre Rawls,NYC,2016,GK,2016-09-15
14381,Andre Rawls,NYC,2017,GK,2017-04-15
14382,Andre Rawls,NYC,2017,GK,2017-09-15
13913,Andre Rawls,NYC,2018,GK,2018-05-01
13914,Andre Rawls,NYC,2018,GK,2018-09-15
13466,Andre Rawls,COL,2019,GK,2019-06-01
13468,Andre Rawls,COL,2019,GK,2019-09-13
11956,Andre Rawls,COL,2020,GK,2020-12-01
11467,Andre Rawls,COL,2021,GK,2021-04-15



goals_added_gk: 0 rows

goals_added: 0 rows

xgoals: 0 rows

xpass: 0 rows

--- Auston Trusty ---

salaries: 11 rows


,Player,Team,Season,Position,Date
12975,Auston Trusty,PHI,2016,D,2016-09-15
10583,Auston Trusty,PHI,2017,D,2017-04-15
10584,Auston Trusty,PHI,2017,D,2017-09-15
10348,Auston Trusty,PHI,2018,D,2018-05-01
10349,Auston Trusty,PHI,2018,D,2018-09-15
9667,Auston Trusty,PHI,2019,D,2019-09-13
9669,Auston Trusty,PHI,2019,D,2019-06-01
5399,Auston Trusty,COL,2020,D,2020-12-01
4873,Auston Trusty,COL,2021,D,2021-04-15
4874,Auston Trusty,COL,2021,D,2021-09-30



goals_added_gk: 0 rows

goals_added: 5 rows


,Player,Team,Season,Position,Minutes
2704,Auston Trusty,PHI,2018,CB,3414
2071,Auston Trusty,PHI,2019,CB,2153
3895,Auston Trusty,COL,2020,CB,586
3555,Auston Trusty,COL,2021,CB,3370
1096,Auston Trusty,COL,2022,CB,1589



xgoals: 5 rows


,Player,Team,Season,Position,Minutes
5574,Auston Trusty,PHI,2018,CB,3414
3878,Auston Trusty,PHI,2019,CB,2153
2936,Auston Trusty,COL,2020,CB,586
4932,Auston Trusty,COL,2021,CB,3370
4254,Auston Trusty,COL,2022,CB,1589



xpass: 5 rows


,Player,Team,Season,Position,Minutes
4697,Auston Trusty,PHI,2018,CB,3414
2282,Auston Trusty,PHI,2019,CB,2153
1148,Auston Trusty,COL,2020,CB,586
1322,Auston Trusty,COL,2021,CB,3370
2952,Auston Trusty,COL,2022,CB,1589


In [2]:
from itscalledsoccer import AmericanSoccerAnalysis

asa = AmericanSoccerAnalysis()

players_to_check = ["Andre Rawls", "Auston Trusty"]

for league in ["mls", "uslc"]:
    for player in players_to_check:
        result = asa.get_players(
            leagues=league,
            names=player
        )
        print(f"\n--- {player} | {league.upper()} ---")
        print(f"Type: {type(result)}")
        print(f"Shape: {result.shape}")

        display(result)


--- Andre Rawls | MLS ---
Type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (1, 13)


birth_date,player_id,secondary_broad_position,secondary_general_position,height_ft,primary_broad_position,weight_lb,height_in,season_name,nationality,primary_general_position,player_name,competition
str,str,str,str,i64,str,i64,i64,object,str,str,str,str
"""1991-12-20""","""NPqxeE4XQ9""",null,null,6,null,190,2,{},"""USA""",null,"""Andre Rawls""","""mls"""



--- Auston Trusty | MLS ---
Type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (1, 13)


birth_date,player_id,secondary_broad_position,secondary_general_position,height_ft,primary_broad_position,weight_lb,height_in,season_name,nationality,primary_general_position,player_name,competition
str,str,str,str,i64,str,i64,i64,object,str,str,str,str
"""1998-08-12""","""vzqo71rBqa""",null,null,6,"""DF""",185,3,"['2018', '2019', '2020', '2021', '2022']","""USA""","""CB""","""Auston Trusty""","""mls"""



--- Andre Rawls | USLC ---
Type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (1, 13)


birth_date,player_id,secondary_broad_position,secondary_general_position,height_ft,primary_broad_position,weight_lb,height_in,season_name,nationality,primary_general_position,player_name,competition
str,str,str,str,i64,str,i64,i64,object,str,str,str,str
"""1991-12-20""","""NPqxeE4XQ9""",null,null,6,"""GK""",190,2,"['2018', '2019', '2021']","""USA""","""GK""","""Andre Rawls""","""uslc"""



--- Auston Trusty | USLC ---
Type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (1, 13)


birth_date,player_id,secondary_broad_position,secondary_general_position,height_ft,primary_broad_position,weight_lb,height_in,season_name,nationality,primary_general_position,player_name,competition
str,str,str,str,i64,str,i64,i64,object,str,str,str,str
"""1998-08-12""","""vzqo71rBqa""",null,null,6,"""DF""",185,3,2017,"""USA""","""CB""","""Auston Trusty""","""uslc"""


In [5]:
# Retrieve MLS and USLC player metadata

all_players = asa.get_players(
    leagues=["mls", "uslc"]
)

print(f"Type: {type(all_players)}")
print(f"Shape: {all_players.shape}")
print(f"Columns: {all_players.columns}")

display(all_players.head(10))

# Inspect player counts and identifier consistency
print("\nPlayers by competition:")
print(
    all_players.group_by("competition")
    .agg(
        pl.len().alias("records"),
        pl.col("player_id").n_unique().alias("unique_players")
    )
)

Type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (7458, 13)
Columns: ['birth_date', 'player_id', 'secondary_broad_position', 'secondary_general_position', 'height_ft', 'primary_broad_position', 'weight_lb', 'height_in', 'season_name', 'nationality', 'primary_general_position', 'player_name', 'competition']


birth_date,player_id,secondary_broad_position,secondary_general_position,height_ft,primary_broad_position,weight_lb,height_in,season_name,nationality,primary_general_position,player_name,competition
str,str,str,str,i64,str,i64,i64,object,str,str,str,str
"""1983-04-03""","""0Oq6006M6D""",null,null,6,null,179,0,{},"""USA""",null,"""Ugo Ihemelu""","""mls"""
"""1983-08-26""","""0Oq60APM6D""",null,null,5,"""DF""",172,11,"['2013', '2014', '2015', '2016', '2017', '2018']","""USA""","""CB""","""Jason Hernandez""","""mls"""
"""1983-08-05""","""0Oq60VgM6D""",null,"""CM""",6,"""MF""",176,1,"['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020']","""USA""","""DM""","""Jeff Larentowicz""","""mls"""
"""2002-06-27""","""0Oq624oPq6""",null,null,5,"""DF""",150,8,"['2024', '2025', '2026']","""USA""","""FB""","""Kalani Kossa-Rienzi""","""mls"""
"""2005-10-11""","""0Oq624zdq6""",null,null,5,"""FW""",165,11,2026,"""Switzerland""","""W""","""Noah Streit""","""mls"""
"""2002-03-28""","""0Oq62GYAq6""",null,null,null,null,null,null,{},"""USA""",null,"""Oscar Herrera""","""mls"""
"""2007-01-24""","""0Oq62JVPq6""",null,null,6,"""DF""",172,1,2026,"""Belgium""","""FB""","""Joyeux Masanka Bungi""","""mls"""
"""2000-07-03""","""0Oq62Y3zq6""",null,null,null,null,null,null,{},"""USA""",null,"""Isaac Walker""","""mls"""
"""1999-07-10""","""0Oq630dXQ6""",null,null,null,"""MF""",null,null,"['2021', '2022', '2023', '2024', '2025']","""France""","""W""","""Kévin Cabral""","""mls"""



Players by competition:
shape: (2, 3)
┌─────────────┬─────────┬────────────────┐
│ competition ┆ records ┆ unique_players │
│ ---         ┆ ---     ┆ ---            │
│ str         ┆ u32     ┆ u32            │
╞═════════════╪═════════╪════════════════╡
│ mls         ┆ 3641    ┆ 3641           │
│ uslc        ┆ 3817    ┆ 3817           │
└─────────────┴─────────┴────────────────┘


In [11]:

players_registry = all_players.to_pandas()

registry_names = (
    players_registry[
        ["player_id", "player_name", "competition"]
    ]
    .drop_duplicates()
)

name_candidates = (
    registry_names
    .groupby("player_name", as_index=False)
    .agg(
        candidate_ids=("player_id", "nunique"),
        player_ids=("player_id", lambda x: sorted(x.unique())),
        competitions=("competition", lambda x: sorted(x.unique()))
    )
)

salary_only_registry = salary_only.merge(
    name_candidates,
    left_on="Player",
    right_on="player_name",
    how="left",
    validate="many_to_one"
)

# Categorize match results
salary_only_registry["match_status"] = "unmatched"

salary_only_registry.loc[
    salary_only_registry["candidate_ids"] == 1,
    "match_status"
] = "unique_match"

salary_only_registry.loc[
    salary_only_registry["candidate_ids"] > 1,
    "match_status"
] = "ambiguous_match"

print("Player-season records by matching status:")
display(
    salary_only_registry["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .reset_index(name="player_seasons")
)

print("\nDistinct players by matching status:")
display(
    salary_only_registry
    .groupby("match_status")["Player"]
    .nunique()
    .reset_index(name="players")
)

print("\nUnmatched or ambiguous examples:")
display(
    salary_only_registry.loc[
        salary_only_registry["match_status"] != "unique_match",
        ["Player", "Season", "candidate_ids", "player_ids", "competitions", "match_status"]
    ].head(20)
)


Player-season records by matching status:


,match_status,player_seasons
0,unique_match,1338
1,ambiguous_match,11



Distinct players by matching status:


,match_status,players
0,ambiguous_match,6
1,unique_match,864



Unmatched or ambiguous examples:


,Player,Season,candidate_ids,player_ids,competitions,match_status
49,Juan Ramírez,2016,2,"[EGMPlRmVMa, NPqxoY2X59]","[mls, uslc]",ambiguous_match
155,Juan Ramírez,2017,2,"[EGMPlRmVMa, NPqxoY2X59]","[mls, uslc]",ambiguous_match
373,João Pedro,2019,2,"[Pk5L1ARPqO, jYQJgRXGMG]","[mls, uslc]",ambiguous_match
386,Luiz Fernando,2019,2,"[gpMOaWD2qz, wvq9gbRaMW]","[mls, uslc]",ambiguous_match
575,Christian Torres,2021,2,"[315VPg0EQ9, jYQJawDyqG]","[mls, uslc]",ambiguous_match
612,Graham Smith,2021,2,"[9vQ2Zyd1QK, eVq3j3oG5W]","[mls, uslc]",ambiguous_match
733,Christian Torres,2022,2,"[315VPg0EQ9, jYQJawDyqG]","[mls, uslc]",ambiguous_match
790,Jonathan Jimenez,2022,2,"[9vQ2ZaBoQK, OlMlexxw5L]","[mls, uslc]",ambiguous_match
889,Christian Torres,2023,2,"[315VPg0EQ9, jYQJawDyqG]","[mls, uslc]",ambiguous_match
936,Jonathan Jimenez,2023,2,"[9vQ2ZaBoQK, OlMlexxw5L]","[mls, uslc]",ambiguous_match


In [12]:

# Identify names associated with multiple ASA player IDs
ambiguous_names = (
    salary_only_registry.loc[
        salary_only_registry["match_status"] == "ambiguous_match",
        "Player"
    ]
    .unique()
)

# Retrieve candidate identities from the ASA registry
ambiguous_candidates = (
    players_registry[
        players_registry["player_name"].isin(ambiguous_names)
    ]
    [
        [
            "player_name",
            "player_id",
            "competition",
            "birth_date",
            "nationality",
            "primary_general_position",
            "season_name",
        ]
    ]
    .sort_values(["player_name", "player_id", "competition"])
)

# Retrieve corresponding salary observations
ambiguous_salaries = (
    asa_tables["salaries"].loc[
        lambda df: df["Player"].isin(ambiguous_names),
        ["Player", "Team", "Season", "Position", "Date"]
    ]
    .sort_values(["Player", "Season", "Date"])
)

print("ASA identity candidates:")
display(ambiguous_candidates)

print("\nSalary records:")
display(ambiguous_salaries)


ASA identity candidates:


,player_name,player_id,competition,birth_date,nationality,primary_general_position,season_name
408,Christian Torres,315VPg0EQ9,mls,2004-04-15,USA,W,2020
4060,Christian Torres,315VPg0EQ9,uslc,2004-04-15,USA,W,"[2021, 2022, 2026]"
6664,Christian Torres,jYQJawDyqG,uslc,1996-10-23,Mexico,FB,"[2018, 2019]"
4586,Graham Smith,9vQ2Zyd1QK,uslc,1994-04-09,USA,DM,"[2018, 2019, 2020]"
2446,Graham Smith,eVq3j3oG5W,mls,1995-11-25,USA,CB,"[2018, 2019, 2020]"
6182,Graham Smith,eVq3j3oG5W,uslc,1995-11-25,USA,CB,"[2018, 2019, 2021, 2022, 2023, 2024, 2025, 2026]"
886,Jonathan Jimenez,9vQ2ZaBoQK,mls,2001-04-25,Mexico,NaN,{}
4576,Jonathan Jimenez,9vQ2ZaBoQK,uslc,2001-04-25,Mexico,CM,2025
5539,Jonathan Jimenez,OlMlexxw5L,uslc,1997-12-17,USA,W,2020
1804,João Pedro,Pk5L1ARPqO,mls,1993-04-03,Portugal,DM,"[2017, 2018]"



Salary records:


,Player,Team,Season,Position,Date
13288,Christian Torres,LAFC,2020,F,2020-12-01
11469,Christian Torres,LAFC,2021,F,2021-04-15
11470,Christian Torres,LAFC,2021,F,2021-09-30
10810,Christian Torres,LAFC,2022,F,2022-04-15
10811,Christian Torres,LAFC,2022,F,2022-09-02
9705,Christian Torres,LAFC,2023,F,2023-04-30
9706,Christian Torres,LAFC,2023,M,2023-09-15
15213,Graham Smith,SKC,2018,D,2018-05-01
15214,Graham Smith,SKC,2018,D,2018-09-15
15022,Graham Smith,SKC,2019,D,2019-06-01
